# Exercises XP: RAG Pipeline with Vector Search and Question Answering
This notebook demonstrates data loading, vectorization, FAISS and ChromaDB search, and question answering using Hugging Face models.

In [ ]:
# Exercise 1: Data Loading and Preparation
!pip install -q faiss-cpu==1.7.4 chromadb==0.3.21 numpy<2
import numpy as np
import pandas as pd
import faiss
import json
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [ ]:
# Load the dataset
path = 'labelled_newscatcher_dataset.csv'  # Update with your actual path
pdf = pd.read_csv(path)
pdf['id'] = pdf.index.astype(str)
display(pdf)
pdf_subset = pdf.head(1000)

In [ ]:
# Exercise 2: Vectorization with Sentence Transformers
from sentence_transformers import InputExample, SentenceTransformer
def example_create_fn(title):
    return InputExample(guid=None, texts=[title], label=0.0)
faiss_train_examples = pdf_subset['title'].apply(example_create_fn).tolist()
model = SentenceTransformer('all-MiniLM-L6-v2')
titles_list = pdf_subset['title'].tolist()
faiss_title_embedding = model.encode(titles_list)
print(len(faiss_title_embedding), len(faiss_title_embedding[0]))

In [ ]:
# Exercise 3: FAISS Indexing and Search
pdf_to_index = pdf_subset
id_index = pdf_to_index['id'].astype(int).values
content_encoded_normalized = np.array(faiss_title_embedding)
faiss.normalize_L2(content_encoded_normalized)
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(content_encoded_normalized.shape[1]))
index_content.add_with_ids(content_encoded_normalized, id_index)
def search_content(query, pdf_to_index, k=3):
    query_vector = model.encode([query])
    faiss.normalize_L2(query_vector)
    scores, ids = index_content.search(query_vector, k)
    results = pdf_to_index[pdf_to_index['id'].astype(int).isin(ids[0])].copy()
    results['similarities'] = scores[0]
    return results
display(search_content('animal', pdf_to_index, k=5))

In [ ]:
# Exercise 4: ChromaDB Collection and Querying
chroma_client = chromadb.Client()
collection_name = 'my_news'
if len(chroma_client.list_collections()) > 0 and collection_name in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection(name=collection_name)
print(f'Creating collection: {collection_name}')
collection = chroma_client.create_collection(name=collection_name)
collection.add(
    documents=pdf_subset['title'][:100].tolist(),
    metadatas=[{'topic': topic} for topic in pdf_subset['topic'][:100].tolist()],
    ids=pdf_subset['id'][:100].tolist()
)
import json
results = collection.query(query_texts=['space'], n_results=10)
print(json.dumps(results, indent=4))

In [ ]:
# Exercise 5: Question Answering with Hugging Face Model
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
model_id = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)
pipe = pipeline('text-generation', model=lm_model, tokenizer=tokenizer, max_new_tokens=512, device_map='auto')
question = "What's the latest news on space development?"
context = ' '.join(results['documents'][0])
prompt_template = f'Relevant context: {context}

 The users question: {question}'
lm_response = pipe(prompt_template)
print(lm_response[0]['generated_text'])

## Experiment: Try different questions and context sizes to see how the model's answers change.